In [1]:
import pandas as pd
import torch
import time
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW 
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"There are {torch.cuda.device_count()} GPU(s) available.")
    print(f"We will use the GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

/home/protein/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


There are 1 GPU(s) available.
We will use the GPU: NVIDIA RTX A5000


In [2]:
train_data = pd.read_csv('PI_nonPI_train_80.csv')
test_data = pd.read_csv('PI_nonPI_test_20.csv')

train_data['sequence'] = train_data['sequence'].apply(lambda x: " ".join(list(x)))
test_data['sequence'] = test_data['sequence'].apply(lambda x: " ".join(list(x)))

train_texts = train_data['sequence'].tolist()
train_labels = train_data['label'].tolist()
test_texts = test_data['sequence'].tolist()
test_labels = test_data['label'].tolist()

print(f"Training samples: {len(train_texts)}")
print(f"Testing samples: {len(test_texts)}")

Training samples: 18101
Testing samples: 4526


In [3]:
# Compute class weights based on training data
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_labels), 
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Class Weights computed: {class_weights}")

Class Weights computed: [0.73159001 1.57949389]


In [4]:
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertForSequenceClassification.from_pretrained("Rostlab/prot_bert", num_labels=2)
model.to(device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Rostlab/prot_bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30, 1024, padding_idx=0)
      (position_embeddings): Embedding(40000, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-29): 30 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((10

In [6]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=250, return_tensors='pt')
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=250, return_tensors='pt')
train_labels_tensor = torch.tensor(train_labels)
test_labels_tensor = torch.tensor(test_labels)

batch_size = 16
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels_tensor)
test_dataset = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
optimizer = AdamW(model.parameters(), lr=1e-6, weight_decay=0.01)
num_epochs = 3

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits

        loss = criterion(logits, b_labels)

        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Average Loss: {avg_train_loss:.4f}")

print(f"Training finished in {time.time() - start_time:.2f} seconds.")

Epoch 1/3 | Average Loss: 0.2863
Epoch 2/3 | Average Loss: 0.0618
Epoch 3/3 | Average Loss: 0.0338
Training finished in 4585.00 seconds.


In [8]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

model.eval()

all_preds = []
all_probs = []
all_labels_actual = []

with torch.no_grad():
    for batch in test_loader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[:, 1] 
        
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels_actual.extend(b_labels.cpu().numpy())

y_true = all_labels_actual
y_pred = all_preds
y_prob = all_probs

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1:.4f}")
print(f"ROC-AUC:     {roc_auc:.4f}")
print(f"MCC:         {mcc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")

Accuracy:    0.9903
Precision:   0.9826
Recall:      0.9867
F1 Score:    0.9847
ROC-AUC:     0.9990
MCC:         0.9776
Sensitivity: 0.9867
Specificity: 0.9919


In [9]:
import os

output_dir = './protbert_finetuned_model'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
model_to_save = model.module if hasattr(model, 'module') else model  # Handle distributed/parallel training
model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)


('./protbert_finetuned_model/tokenizer_config.json',
 './protbert_finetuned_model/special_tokens_map.json',
 './protbert_finetuned_model/vocab.txt',
 './protbert_finetuned_model/added_tokens.json')

In [10]:
import os
import sys
import logging
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)

class ProteinInferenceDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = " ".join(list(self.sequences[idx]))
        
        encoding = self.tokenizer(
            seq,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        logger.warning("Using CPU")
    return device

def load_model(model_dir, device):
    if not os.path.exists(model_dir):
        logger.error(f"Model directory not found: {model_dir}")
        sys.exit(1)

    logger.info(f"Loading model from: {model_dir}")
    try:
        tokenizer = BertTokenizer.from_pretrained(model_dir)
        model = BertForSequenceClassification.from_pretrained(model_dir)
        model.to(device)
        model.eval()
        return tokenizer, model
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        sys.exit(1)

# ADDED max_length ARGUMENT HERE
def run_inference(input_csv, output_csv, model_dir, max_length=256, batch_size=32):
    device = get_device()
    tokenizer, model = load_model(model_dir, device)

    if not os.path.exists(input_csv):
        logger.error(f"Input file not found: {input_csv}")
        return

    logger.info(f"Reading input file: {input_csv}")
    try:
        df = pd.read_csv(input_csv)
    except Exception as e:
        logger.error(f"Error reading CSV: {e}")
        return

    required_cols = ['header', 'sequence']
    if not all(col in df.columns for col in required_cols):
        logger.error(f"CSV missing required columns. Found: {df.columns.tolist()}, Expected: {required_cols}")
        return

    ids = df['header'].astype(str).tolist()
    raw_sequences = df['sequence'].astype(str).tolist()
    logger.info(f"Found {len(raw_sequences)} sequences.")
    logger.info(f"Using Max Sequence Length: {max_length}")

    # PASSING THE ARGUMENT HERE
    dataset = ProteinInferenceDataset(raw_sequences, tokenizer, max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    logger.info("Starting inference...")
    all_preds = []
    all_probs = []
    
    count = 0
    total = len(raw_sequences)

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)

            preds = torch.argmax(probs, dim=1).cpu().numpy()
            pos_probs = probs[:, 1].cpu().numpy()

            all_preds.extend(preds)
            all_probs.extend(pos_probs)
            
            count += len(input_ids)
            if count % (batch_size * 10) == 0:
                logger.info(f"Processed {count}/{total} sequences")

    results_df = pd.DataFrame({
        'header': ids,
        'sequence': raw_sequences,
        'prediction': all_preds,
        'probability': all_probs
    })

    results_df.to_csv(output_csv, index=False)
    logger.info(f"Results saved to: {output_csv}")

if __name__ == "__main__":
    MODEL_DIRECTORY = './protbert_finetuned_model'
    INPUT_FILE = 'MAPH_permissive_noPfam_250AA_sequences.csv'
    OUTPUT_FILE = 'predictions_prot_bert_epoch3.csv'
    
    # You can change the max_length here (e.g., 256, 512, 1024)
    run_inference(INPUT_FILE, OUTPUT_FILE, MODEL_DIRECTORY, max_length=256)

2026-01-06 21:38:27,904 - INFO - Using GPU: NVIDIA RTX A5000
2026-01-06 21:38:27,904 - INFO - Loading model from: ./protbert_finetuned_model
2026-01-06 21:38:29,067 - INFO - Reading input file: MAPH_permissive_noPfam_250AA_sequences.csv
2026-01-06 21:38:29,074 - INFO - Found 437 sequences.
2026-01-06 21:38:29,075 - INFO - Using Max Sequence Length: 256
2026-01-06 21:38:29,076 - INFO - Starting inference...
2026-01-06 21:38:39,005 - INFO - Processed 320/437 sequences
2026-01-06 21:38:42,628 - INFO - Results saved to: predictions_prot_bert_epoch3.csv


In [12]:
import pandas as pd
import sys

def count_predictions(file_path):
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        sys.exit(1)

    if 'prediction' not in df.columns:
        sys.exit(1)

    counts = df['prediction'].value_counts().sort_index()
    total = len(df)

    negatives = counts.get(0, 0)
    positives = counts.get(1, 0)

    print(f"Total Sequences: {total}")
    print("-" * 30)
    print(f"Negative (0): {negatives} ({negatives/total*100:.2f}%)")
    print(f"Positive (1): {positives} ({positives/total*100:.2f}%)")

if __name__ == "__main__":
    INPUT_FILE = 'predictions_prot_bert_epoch3.csv'
    count_predictions(INPUT_FILE)

Total Sequences: 437
------------------------------
Negative (0): 279 (63.84%)
Positive (1): 158 (36.16%)
